# GraphBAN 药物-蛋白相互作用预测（UC 关键治疗药物筛选）

**流程**：克隆官方 GraphBAN 仓库 → 安装依赖 → 用预训练模型（BioSNAP inductive）对 候选药物 × 关键蛋白 进行相互作用概率预测 → 筛选概率 > 0.5 的药物-基因对 → 生成 Cytoscape 网络文件与预览图。

**使用方法**：
1. 菜单栏 `代码执行程序 → 更改运行时类型`，选择 **GPU（T4）**；
2. 按顺序运行所有单元格（`代码执行程序 → 全部运行`）；
3. 运行到「上传数据」单元格时，选择本地的 `GraphBAN_input.csv` 上传；
4. 最后一个单元格会自动打包下载全部结果（概率表、筛选结果、Cytoscape 文件、网络预览图）；
5. 将 `cytoscape_edges.csv` 导入 Cytoscape（`File → Import → Network from File`，source=drug，target=gene）即可构建 Figure 9B 网络，节点属性可导入 `cytoscape_nodes.csv`。

> 参考：GraphBAN 官方仓库 https://github.com/HamidHadipour/GraphBAN （Hadipour et al., Nature Communications, 2025）

In [ ]:
# ===== 1. 克隆官方仓库并安装依赖（约 5-10 分钟）=====
# 注意：仓库 requirements.txt 钉死了 CUDA 11.8 / 旧版本（dgl+ cu118、pyg-lib+pt23cu118、numpy 1.21 等），
# 与 Colab 当前环境（Python 3.12、torch 2.x、cu12x）不兼容，因此按包名安装最新兼容版。
import os, sys, subprocess, site

if not os.path.exists('GraphBAN'):
    !git clone https://github.com/HamidHadipour/GraphBAN

# 1) 其余依赖按最新兼容版安装（torch 用 Colab 自带的，不动）
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch_geometric', 'torchmetrics', 'dgllife', 'rdkit', 'prettytable',
    'yacs', 'pyarrow', 'transformers', 'fair-esm', 'networkx', 'matplotlib'], check=True)

# 2) 按 Colab 实际的 torch/CUDA 版本从 DGL 官方 wheel 源安装 dgl（带兜底）
import torch
tv = torch.__version__.split('+')[0].split('.')
torch_mm = f'{tv[0]}.{tv[1]}'
cuda = torch.version.cuda
candidates = []
if cuda:
    candidates.append(f'https://data.dgl.ai/wheels/torch-{torch_mm}/cu{cuda.replace(".", "")}/repo.html')
candidates.append(f'https://data.dgl.ai/wheels/torch-{torch_mm}/repo.html')
print(f'torch={torch.__version__}, cuda={cuda}')
for url in candidates:
    print('尝试 DGL 源:', url)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dgl', '-f', url])
    if r.returncode == 0:
        break
else:
    print('DGL 官方源无匹配 wheel，回退到 PyPI 版 dgl')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dgl'], check=True)

# 3) dgl 的 graphbolt 子模块依赖编译版 C++ 库和旧版 torchdata，与 Colab 新环境不兼容；
#    GraphBAN 完全用不到 graphbolt —— 若 import dgl 失败，直接把 graphbolt/__init__.py 替换为存根
try:
    import dgl
    print('dgl 可直接导入')
except Exception as e:
    print(f'dgl 导入失败（{type(e).__name__}），将 graphbolt 替换为存根...')
    gb = os.path.join(site.getsitepackages()[0], 'dgl', 'graphbolt', '__init__.py')
    with open(gb, 'w') as f:
        f.write('''# stub: GraphBAN 不使用 graphbolt，此处仅为通过 dgl 的导入检查
def __getattr__(name):
    class _Dummy:
        def __init__(self, *a, **k): pass
        def __call__(self, *a, **k): return self
        def __iter__(self): return iter(())
    return _Dummy
''')
    import dgl
    print('graphbolt 存根完成')

import dgllife, rdkit, transformers, esm
print('环境准备完成，dgl 版本:', dgl.__version__)

In [ ]:
# ===== 2. 确认预训练模型权重存在 =====
import os, glob

MODEL_PATH = 'GraphBAN/inductive_mode/trained_models/biosnap/inductive/seed12/result/best_model_epoch_45.pth'

if os.path.exists(MODEL_PATH):
    print('找到预训练模型:', MODEL_PATH)
else:
    print('未找到默认路径的权重，仓库内可用的 .pth 文件如下，请修改 MODEL_PATH：')
    for p in glob.glob('GraphBAN/inductive_mode/trained_models/**/*.pth', recursive=True):
        print('  ', p)
    raise FileNotFoundError(MODEL_PATH)

In [ ]:
# ===== 3. 上传输入数据并生成 GraphBAN 输入文件 =====
# 只需上传本地的 GraphBAN_input.csv 
from google.colab import files
import pandas as pd

uploaded = files.upload()

src = pd.read_csv('GraphBAN_input.csv')
assert list(src.columns) == ['drug', 'smiles', 'protein', 'sequence'], src.columns
assert len(src) == 600 and src['smiles'].notna().all() and src['sequence'].notna().all()

# GraphBAN 要求表头为 SMILES, Protein, Y（Y 为占位标签，纯预测填 0）
test_df = pd.DataFrame({'SMILES': src['smiles'], 'Protein': src['sequence'], 'Y': 0})
test_df.to_csv('graphban_test.csv', index=False)

# 行序对应的药物名/基因名映射，用于结果回填
pair_df = src[['drug', 'protein', 'smiles']].rename(columns={'protein': 'gene'})
pair_df.to_csv('pair_info.csv', index=False)

print('输入校验通过：', test_df.shape, '| 药物数:', pair_df['drug'].nunique(), '| 基因:', sorted(pair_df['gene'].unique()))

In [ ]:
# ===== 4. 运行 GraphBAN 推理（强制 CPU + 修补权重加载）=====
# 必须从仓库根目录运行（脚本用相对路径读取 inductive_mode/GraphBAN_DA.yaml），
# 输出固定写到 GraphBAN/inductive_mode/result/ 目录下
import os, shutil

os.makedirs('GraphBAN/inductive_mode/result', exist_ok=True)

# 修补 predict.py：权重在 GPU 上保存，CPU 加载必须指定 map_location
pred_py = 'GraphBAN/inductive_mode/predict.py'
src = open(pred_py).read()
old = 'modelG.load_state_dict(torch.load(args.trained_model))'
new = 'modelG.load_state_dict(torch.load(args.trained_model, map_location=device))'
if old in src:
    open(pred_py, 'w').write(src.replace(old, new))
    print('已修补 predict.py（map_location=device）')
else:
    print('predict.py 已是修补过的版本')

# CUDA_VISIBLE_DEVICES="" 让 torch 看不到 GPU → 全部走 CPU（600 对数据量小，足够快）
!cd GraphBAN && CUDA_VISIBLE_DEVICES="" python inductive_mode/predict.py \
    --test_path ../graphban_test.csv \
    --trained_model inductive_mode/trained_models/biosnap/inductive/seed12/result/best_model_epoch_45.pth \
    --save_dir graphban_predictions.csv

shutil.copy('GraphBAN/inductive_mode/result/graphban_predictions.csv', 'graphban_predictions.csv')
print('推理完成，结果已复制到 graphban_predictions.csv')

In [ ]:
# ===== 5. 整理预测结果，回填药物名/基因名 =====
import pandas as pd

pred = pd.read_csv('graphban_predictions.csv')
print('预测结果列名：', list(pred.columns))
#assert len(pred) == 600, f'输出行数 {len(pred)} != 600，请检查'

# 自动识别概率列（名称含 prob/score/pred 的列；否则取最后一列数值列）
PROB_COL = None  # 如需手动指定，填列名字符串
if PROB_COL is None:
    cands = [c for c in pred.columns if any(k in c.lower() for k in ('prob', 'score', 'pred'))]
    num_cols = [c for c in pred.columns if pd.api.types.is_numeric_dtype(pred[c])]
    PROB_COL = cands[0] if cands else num_cols[-1]
print('使用概率列：', PROB_COL)

# 按行顺序回填 drug / gene（predict.py 不打乱行序）
res = pair_df.copy()
res['probability'] = pred[PROB_COL].values
res = res[['drug', 'gene', 'smiles', 'probability']].sort_values('probability', ascending=False)
res.to_csv('GraphBAN_all_predictions.csv', index=False)
print('全部 600 对概率已保存：GraphBAN_all_predictions.csv')
res.head(10)

In [ ]:
# ===== 6. 按概率 > 0.5 筛选关键药物-基因对 =====
THRESHOLD = 0.5  # 需求阈值；如需更严格可上调（如 0.7），如需放宽可下调

key = res[res['probability'] > THRESHOLD].copy()
key.to_csv('GraphBAN_key_pairs.csv', index=False)

print(f'阈值 > {THRESHOLD}：筛出 {len(key)} 对药物-基因相互作用')
print('涉及药物数：', key['drug'].nunique(), '| 涉及基因：', sorted(key['gene'].unique()))
print('\n各基因命中数：')
print(key.groupby('gene').size())
print('\n各基因最高概率对：')
print(key.groupby('gene').apply(lambda d: d.nlargest(3, 'probability'), include_groups=False))

if len(key) == 0:
    print('\n⚠ 无超过阈值的对，可查看 GraphBAN_all_predictions.csv 中的概率分布，适当下调 THRESHOLD')
elif key['drug'].nunique() > 50:
    print('\n⚠ 命中药物较多，如需更严格可上调 THRESHOLD 后重跑本单元格')

In [ ]:
# ===== 7. 生成 Cytoscape 网络文件 =====
# 边表：source=药物, target=基因
edges = key.rename(columns={'drug': 'source', 'gene': 'target'})
edges['interaction'] = 'interacts_with'
edges = edges[['source', 'target', 'interaction', 'probability', 'smiles']]
edges.to_csv('cytoscape_edges.csv', index=False)

# 节点表：节点名 + 类型（drug / gene）
nodes = pd.concat([
    pd.DataFrame({'node': key['drug'].unique(), 'type': 'drug'}),
    pd.DataFrame({'node': key['gene'].unique(), 'type': 'gene'}),
], ignore_index=True)
nodes.to_csv('cytoscape_nodes.csv', index=False)

print('cytoscape_edges.csv :', len(edges), '条边')
print('cytoscape_nodes.csv :', len(nodes), '个节点（drug: %d, gene: %d）' % ((nodes['type']=='drug').sum(), (nodes['type']=='gene').sum()))
print()
print('Cytoscape 导入步骤：')
print('  1. File → Import → Network from File，选择 cytoscape_edges.csv')
print('  2. 设置 source 列 = source，target 列 = target，interaction 列 = interaction')
print('  3. File → Import → Table from File 导入 cytoscape_nodes.csv（Key = node），可按 type 设置节点颜色/形状')
print('  4. 推荐布局：Layout → Group Attributes Layout → type（药物与基因分两侧）')

In [ ]:
# ===== 8. 网络预览图（快速查看用，正式 Figure 9B 请在 Cytoscape 中美化）=====
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx

if len(key) > 0:
    G = nx.Graph()
    for _, r in key.iterrows():
        G.add_edge(r['drug'], r['gene'], weight=r['probability'])

    genes = sorted(key['gene'].unique())
    drugs = sorted(key['drug'].unique())
    pos = {}
    for i, g in enumerate(genes):
        pos[g] = (0, i)
    for i, d in enumerate(drugs):
        pos[d] = (1, i * max(1, len(genes)) / max(1, len(drugs)))

    plt.figure(figsize=(12, max(6, len(drugs) * 0.35)))
    nx.draw_networkx_edges(G, pos, alpha=0.4, edge_color='gray')
    nx.draw_networkx_nodes(G, pos, nodelist=genes, node_color='#E15759', node_shape='s', node_size=600, label='gene')
    nx.draw_networkx_nodes(G, pos, nodelist=drugs, node_color='#4E79A7', node_size=300, label='drug')
    nx.draw_networkx_labels(G, pos, font_size=7)
    plt.legend(scatterpoints=1)
    plt.title(f'Key drug-gene interaction network (probability > {THRESHOLD})')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('network_preview.png', dpi=200)
    plt.show()
    print('预览图已保存：network_preview.png')
else:
    print('无超过阈值的对，跳过预览图')

In [ ]:
# ===== 9. 打包并下载全部结果 =====
import os, zipfile
from google.colab import files

out_files = [
    'GraphBAN_all_predictions.csv',   # 全部 600 对概率
    'GraphBAN_key_pairs.csv',         # 概率 > 阈值的关键药物-基因对
    'cytoscape_edges.csv',            # Cytoscape 边表
    'cytoscape_nodes.csv',            # Cytoscape 节点表
    'network_preview.png',            # 网络预览图
]

with zipfile.ZipFile('GraphBAN_results.zip', 'w') as z:
    for f in out_files:
        if os.path.exists(f):
            z.write(f)
            print('打包:', f)

files.download('GraphBAN_results.zip')
print('\n完成！将 zip 解压到本地 GraphBAN 文件夹即可。')